In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="DVnHjxknEUnHNOnqYh9H")
project = rf.workspace("deankt-fans").project("brick-t6ata")
version = project.version(2)
dataset = version.download("yolov8")
                
                
                

print(dataset.location)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Brick-2 in yolov8:: 100%|██████████| 87/87 [00:00<00:00, 4841.96it/s]

d:\Amr\Portfolio\All time project\ready to publish\CV\Project 7\Brick-2


In [1]:
from ultralytics import RTDETR
import warnings
warnings.filterwarnings('ignore')

# load base model RT-DETR (pretrained COCO) buat transfer learning
model = RTDETR("rtdetr-l.pt")

model.train(
    data=r"D:/Amr/Portfolio/All time project/ready to publish/CV/Project 7/Brick-4/data.yaml",
    epochs=50,
    imgsz=256,
    batch=16,
    patience=10,
)

New https://pypi.org/project/ultralytics/8.4.130 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.115  Python-3.13.3 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:/Amr/Portfolio/All time project/ready to publish/CV/Project 7/Brick-4/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=256, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000001EDC6B50130>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

In [6]:
import cv2
from ultralytics import RTDETR

# ── CONFIG ────────────────────────────────────────────────────────────
VIDEO_PATH = r"D:/Amr/Portfolio/All time project/ready to publish/CV/Project 7/source.mp4"        # ganti ke 0 untuk webcam realtime
MODEL_PATH = r"D:/Amr/Portfolio/All time project/ready to publish/CV/Project 7/runs/detect/train-16/weights/best.pt"                 # ganti ke model custom kamu (mis. best.pt) untuk deteksi bata
OUTPUT_PATH = "conveyor_output.mp4"

LINE_START = (290, 670)
LINE_END = (520, 630) 

CONF_THRESHOLD = 0.5
# ─────────────────────────────────────────────────────────────────────

model = RTDETR(MODEL_PATH)

cap = cv2.VideoCapture(VIDEO_PATH)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS) or 30

writer = cv2.VideoWriter(OUTPUT_PATH, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

prev_side = {}
counted_ids = set()
count = 0


def side_of_line(px, py):
    x1, y1 = LINE_START
    x2, y2 = LINE_END
    return (x2 - x1) * (py - y1) - (y2 - y1) * (px - x1)


# stream=True + persist=True -> ByteTrack tracking 
results = model.track(
    source=VIDEO_PATH,
    conf=CONF_THRESHOLD,
    tracker="bytetrack.yaml",
    persist=True,
    stream=True,
)

for r in results:
    frame = r.orig_img.copy()

    if r.boxes is not None and r.boxes.id is not None:
        boxes = r.boxes.xyxy.cpu().numpy()
        ids = r.boxes.id.cpu().numpy().astype(int)

        for box, track_id in zip(boxes, ids):
            x1, y1, x2, y2 = box
            cx, cy = int((x1 + x2) / 2), int((y1 + y2) / 2)  

            curr_side = side_of_line(cx, cy)

            if track_id in prev_side:
                if prev_side[track_id] * curr_side < 0 and track_id not in counted_ids:
                    count += 1
                    counted_ids.add(track_id)

            prev_side[track_id] = curr_side

            cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
            cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)
            cv2.putText(frame, f"ID {track_id}", (int(x1), int(y1) - 8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    # Calculator line
    cv2.line(frame, LINE_START, LINE_END, (255, 0, 0), 3)

    cv2.putText(frame, f"Count: {count}", (30, 60),
                cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 255), 3)

    writer.write(frame)
    cv2.imshow("Conveyor Counter", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
writer.release()
cv2.destroyAllWindows()
print(f"Object counted: {count}")
print(f"Video saved at: {OUTPUT_PATH}")


video 1/1 (frame 1/135) D:\Amr\Portfolio\All time project\ready to publish\CV\Project 7\source.mp4: 256x256 11 Bricks, 11.1ms
video 1/1 (frame 2/135) D:\Amr\Portfolio\All time project\ready to publish\CV\Project 7\source.mp4: 256x256 11 Bricks, 14.0ms
video 1/1 (frame 3/135) D:\Amr\Portfolio\All time project\ready to publish\CV\Project 7\source.mp4: 256x256 10 Bricks, 13.6ms
video 1/1 (frame 4/135) D:\Amr\Portfolio\All time project\ready to publish\CV\Project 7\source.mp4: 256x256 9 Bricks, 11.3ms
video 1/1 (frame 5/135) D:\Amr\Portfolio\All time project\ready to publish\CV\Project 7\source.mp4: 256x256 10 Bricks, 11.7ms
video 1/1 (frame 6/135) D:\Amr\Portfolio\All time project\ready to publish\CV\Project 7\source.mp4: 256x256 12 Bricks, 12.1ms
video 1/1 (frame 7/135) D:\Amr\Portfolio\All time project\ready to publish\CV\Project 7\source.mp4: 256x256 11 Bricks, 13.2ms
video 1/1 (frame 8/135) D:\Amr\Portfolio\All time project\ready to publish\CV\Project 7\source.mp4: 256x256 10 Bricks,